💡 **Environment:** `clamp-analyses`

# ARCHS4 technical noise analysis

We screened gene loadings (**Z**) and sample scores (**B**) of the ARCHS4 CLAMPfull canonical latent variables (LVs) for technical associations.

## Z: gene features

- Transcript length, log10-transformed
- Transcript GC percentage

Each feature is correlated with LV loadings across all genes and again among genes with nonzero loading. **Z flag threshold:** at least one Pearson test with |r| ≥ 0.20, BH-adjusted FDR < 0.20, and absolute robust z-score ≥ 4 across LVs.

## B: sample features

Categorical GEO metadata:

- Platform
- Instrument model
- Library selection
- Library source
- Library strategy
- Molecule
- Submission year

Numeric QC:

- log10 library size
- log10 detected genes
- Mitochondrial read fraction
- Ribosomal read fraction
- Protein-coding read fraction
- Single-cell probability

All listed B features enter one linear regression for each LV. Study identity is excluded. The regression is scored with 5-fold cross-validation: samples are shuffled with a fixed seed and split into five folds; for each fold, imputation, scaling, reference levels and coefficients are fit on the other four folds and the held-out fold is predicted. R² is computed once on the pooled held-out predictions. **B flag threshold:** cross-validated joint technical R² ≥ 0.25.

## Result filters

- CLAMP pathway annotation: AUC > 0.70 and FDR < 0.05
- Trait association: FDR < 0.05
- Both filters: the same LV passes both criteria

The tables show Z only, B only, Z and B, and Z or B.

## Seeds and threshold

- Flags are also computed for the nine stability-seed models (`seeds/`), each with its own 1,728 LVs; **main** is the model used for all tables above. Per-seed counts are Z and B flags only: pathway and trait filters are applied to the main model.
- The B threshold (R² ≥ 0.25) sits in the tail of the R² distribution: the median LV has R² ≈ 0.04 and about 3% of LV-models (550 of 17,280) pass.


In [ ]:
suppressPackageStartupMessages(library(data.table))
groups_file <- snakemake@input[["groups"]]
affected_file <- snakemake@input[["affected"]]


## Flag summary

In [ ]:
group_summary <- as.data.frame(fread(groups_file))
group_summary

## Affected LVs

In [ ]:
affected_lvs <- fread(affected_file)[, .(LV, affected_group, gene_flag, sample_flag,
  technical_joint, n_pathways, n_traits, has_annotation, has_trait, has_both_filters)]
setorder(affected_lvs, affected_group, LV)
affected_lvs <- as.data.frame(affected_lvs)
options(repr.matrix.max.rows = 400, repr.matrix.max.cols = 12)
affected_lvs

## Flags across seeds

In [ ]:
tn_dir <- dirname(groups_file)
seed_dirs <- c(main = tn_dir, setNames(list.dirs(file.path(tn_dir, "seeds"), recursive = FALSE),
  basename(list.dirs(file.path(tn_dir, "seeds"), recursive = FALSE))))
read_model <- function(d) {
  b <- fread(file.path(d, "lv_sample_metadata_canonical.csv"))[covariate == "technical_joint", .(LV, r2 = value)]
  z <- fread(file.path(d, "lv_gene_properties_canonical.csv"))[, .(LV, gene_flag)]
  merge(b, z, by = "LV")[, sample_flag := r2 >= 0.25]
}
seed_lvs <- rbindlist(lapply(seed_dirs, read_model), idcol = "seed")
seed_lvs[, seed := factor(seed, levels = names(seed_dirs))]
seed_summary <- seed_lvs[, .(n_lvs = .N,
  Z_only = sum(gene_flag & !sample_flag), B_only = sum(!gene_flag & sample_flag),
  Z_and_B = sum(gene_flag & sample_flag), Z_or_B = sum(gene_flag | sample_flag),
  median_r2 = round(median(r2), 3), max_r2 = round(max(r2), 3)), by = seed]
as.data.frame(seed_summary)

## Technical R² distribution

In [ ]:
suppressPackageStartupMessages(library(ggplot2))
n_all <- nrow(seed_lvs)
n_pass <- sum(seed_lvs$sample_flag)
n_flag <- seed_lvs[, .(n = sum(sample_flag)), by = seed]
lab <- setNames(sprintf("%s: %d flagged", n_flag$seed, n_flag$n), n_flag$seed)
ggplot(seed_lvs, aes(r2)) +
  geom_histogram(binwidth = 0.02, boundary = 0, fill = "grey60", colour = "white") +
  geom_vline(xintercept = 0.25, colour = "firebrick", linetype = "dashed") +
  scale_y_continuous(trans = "pseudo_log", breaks = c(0, 1, 10, 100, 1000)) +
  facet_wrap(~seed, ncol = 5, labeller = as_labeller(lab)) +
  labs(x = "R² (B matrix)", y = "LVs (pseudo-log scale)",
       title = sprintf("Distribution of technical R² across LVs, all models (dashed: 0.25; %d of %d LV-models flagged)", n_pass, n_all)) +
  theme_bw(base_size = 11)

In [ ]:
ggplot(seed_lvs, aes(r2)) +
  geom_histogram(binwidth = 0.02, boundary = 0, fill = "grey60", colour = "white") +
  geom_vline(xintercept = 0.25, colour = "firebrick", linetype = "dashed") +
  scale_y_continuous(trans = "pseudo_log", breaks = c(0, 1, 10, 100, 1000, 10000)) +
  labs(x = "R² (B matrix)", y = "LVs (pseudo-log scale)",
       title = sprintf("All LVs pooled across models (dashed: 0.25; %d of %d flagged)", n_pass, n_all)) +
  theme_bw(base_size = 11)

In [ ]:
ggplot(seed_lvs, aes(r2)) +
  geom_density(aes(group = seed), colour = "grey70", linewidth = 0.4) +
  geom_density(colour = "black", linewidth = 0.9) +
  geom_vline(xintercept = 0.25, colour = "firebrick", linetype = "dashed") +
  annotate("text", x = 0.27, y = Inf, hjust = 0, vjust = 2, colour = "firebrick",
           label = sprintf("%d of %d flagged", n_pass, n_all)) +
  coord_cartesian(xlim = c(0, 0.65)) +
  labs(x = "R² (B matrix)", y = "Density",
       title = "Density of technical R² (black: all LVs pooled; grey: each model; dashed: 0.25)") +
  theme_bw(base_size = 11)